In [1]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import re
import pandas as pd

from IPython.display import display
from googleapiclient.discovery import build
from transformers import pipeline

import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")

# =========================================================
# PART 1 — DATA EXTRACTION & PREPROCESSING
# =========================================================

# -------------------------
# YouTube API Setup
# -------------------------

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY
)

# YouTube video ID
video_id = "f_lRdkH_QoY"


# -------------------------
# Fetch YouTube Comments
# -------------------------

comments = []

request = youtube.commentThreads().list(
    part="snippet",
    videoId=video_id,
    maxResults=100,
    textFormat="plainText"
)

response = request.execute()

while request:

    for item in response["items"]:

        comment_data = item["snippet"]["topLevelComment"]["snippet"]

        # number of replies
        replies = item["snippet"]["totalReplyCount"]

        # cleaner date format
        formatted_date = pd.to_datetime(
            comment_data["publishedAt"]
        ).strftime("%m/%d/%Y")

        comments.append([
            comment_data["textOriginal"],
            comment_data["likeCount"],
            replies,
            formatted_date
        ])

    # move to next page
    request = youtube.commentThreads().list_next(
        request,
        response
    )

    if request:
        response = request.execute()


# -------------------------
# Create Raw DataFrame
# -------------------------

df_raw = pd.DataFrame(
    comments,
    columns=[
        "comment",
        "likes",
        "replies",
        "date"
    ]
)

print("TOTAL COMMENTS:")
print(len(df_raw))


# =========================================================
# RAW COMMENTS SAMPLE
# =========================================================

print("\n=========== RAW COMMENTS SAMPLE ===========\n")

# skip first long timestamp comment
raw_sample = df_raw["comment"].iloc[1:11]

for i, comment in enumerate(raw_sample, start=2):

    # shorten long comments
    short_comment = comment[:120]

    if len(comment) > 120:
        short_comment += "..."

    print(f"{i}. {short_comment}")
    print()


# -------------------------
# Extract Timestamps
# -------------------------

def extract_timestamp(comment):

    # supports:
    # 1:23
    # 01:23
    # 1:02:15

    pattern = r'(?<!\d)(\d{1,2}:\d{2}(?::\d{2})?)(?!\d)'

    match = re.search(pattern, comment)

    if match:

        timestamp = match.group(1)

        # ignore real-world clock times
        invalid_words = [
            "am",
            "pm",
            "mst",
            "est",
            "utc"
        ]

        lowered = comment.lower()

        for word in invalid_words:

            if timestamp + " " + word in lowered:
                return None

        return timestamp

    return None


df_raw["timestamp"] = df_raw["comment"].apply(
    extract_timestamp
)


# -------------------------
# Keep Timestamp Comments
# -------------------------

df_extracted = df_raw.dropna(
    subset=["timestamp"]
).copy()

print("\nCOMMENTS WITH TIMESTAMPS:")
print(len(df_extracted))


# -------------------------
# Convert Timestamp to Seconds
# -------------------------

def timestamp_to_seconds(timestamp):

    parts = timestamp.split(":")

    # mm:ss
    if len(parts) == 2:

        minutes = int(parts[0])
        seconds = int(parts[1])

        return minutes * 60 + seconds

    # hh:mm:ss
    elif len(parts) == 3:

        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = int(parts[2])

        return (
            hours * 3600
            + minutes * 60
            + seconds
        )

    return 0


df_extracted["seconds"] = df_extracted["timestamp"].apply(
    timestamp_to_seconds
)


# =========================================================
# DATA DISPLAY CONFIGURATION
# =========================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", 120)


print("\n=========== EXTRACTED DATA ===========\n")

display(
    df_extracted[
        [
            "comment",
            "likes",
            "replies",
            "timestamp",
            "seconds",
            "date"
        ]
    ].head(10)
)


# =========================================================
# PART 2 — DATA CLEANING & PREPARATION
# =========================================================

def clean_comment(text):

    # lowercase
    text = text.lower()

    # remove links
    text = re.sub(r'http\S+', '', text)

   # remove timestamps
    text = re.sub(
        r'\b\d{1,2}:\d{2}(?::\d{2})?\b',
        '',
        text
    )

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    text = text.strip()

    return text


# create cleaned dataframe
df_cleaned = df_extracted.copy()

df_cleaned["clean_comment"] = df_cleaned[
    "comment"
].apply(clean_comment)


# -------------------------
# Remove Weak / Noisy Data
# -------------------------

# remove empty comments
df_cleaned = df_cleaned[
    df_cleaned["clean_comment"].str.len() > 2
]

# remove duplicate comments
df_cleaned = df_cleaned.drop_duplicates(
    subset=["clean_comment"]
)

# remove very long comments
df_cleaned = df_cleaned[
    df_cleaned["clean_comment"].str.len() < 500
]

# remove very short comments
df_cleaned = df_cleaned[
    df_cleaned["clean_comment"].str.split().str.len() >= 2
]


# =========================================================
# COMMENT WEIGHTING SYSTEM
# =========================================================

# higher likes/replies
# = more important comment

df_cleaned["weight"] = (
    (df_cleaned["likes"] * 2)
    +
    (df_cleaned["replies"] * 3)
)

# sort comments by importance
df_cleaned = df_cleaned.sort_values(
    by="weight",
    ascending=False
)


print("\n=========== CLEANED DATA ===========\n")

display(
    df_cleaned[
        [
            "comment",
            "clean_comment",
            "likes",
            "replies",
            "timestamp",
            "seconds",
            "date"
        ]
    ].head(10)
)


# =========================================================
# PART 3 — SENTIMENT ANALYSIS MODEL
# =========================================================

# -------------------------
# Load Hugging Face Model
# -------------------------

classifier = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)


# -------------------------
# Predict Sentiment
# -------------------------

def predict_sentiment(text):

    result = classifier(
        text,
        truncation=True,
        max_length=512
    )[0]

    return pd.Series([
        result["label"].lower(),
        result["score"]
    ])


df_model = df_cleaned.copy()

df_model[
    ["label", "confidence"]
] = df_model["clean_comment"].apply(
    predict_sentiment
)

# round confidence values
df_model["confidence"] = df_model["confidence"].round(2)


# =========================================================
# SENTIMENT SCORE SYSTEM
# =========================================================

sentiment_map = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
}

df_model["sentiment_score"] = df_model[
    "label"
].map(sentiment_map)


# =========================================================
# CREATE TIME SEGMENTS
# =========================================================

# each segment = 30 seconds

df_model["segment"] = (
    df_model["seconds"] // 30
).astype(int)


print("\n=========== FINAL MODEL OUTPUT ===========\n")

display(
    df_model[
        [
            "comment",
            "clean_comment",
            "likes",
            "replies",
            "timestamp",
            "seconds",
            "date",
            "label",
            "confidence",
            "sentiment_score",
            "segment"
        ]
    ].head(15)
)


# =========================================================
# TIMESTAMP FREQUENCY ANALYSIS
# =========================================================

print("\n=========== MOST MENTIONED TIMESTAMPS ===========\n")

timestamp_counts = df_model[
    "timestamp"
].value_counts().head(10)

display(timestamp_counts.to_frame())


# =========================================================
# PEAK DETECTION
# =========================================================

print("\n=========== PEAK DETECTION ===========\n")

# group comments by segment
peak_data = df_model.groupby("segment").agg({
    "comment": "count",
    "likes": "sum",
    "replies": "sum",
    "sentiment_score": "mean"
}).reset_index()

# rename columns
peak_data.columns = [
    "segment",
    "comment_count",
    "total_likes",
    "total_replies",
    "avg_sentiment"
]

# create segment start/end times
peak_data["start_second"] = peak_data["segment"] * 30
peak_data["end_second"] = peak_data["start_second"] + 29

# convert seconds to MM:SS format
def seconds_to_time(seconds):

    minutes = seconds // 60
    seconds = seconds % 60

    return f"{minutes:02d}:{seconds:02d}"

# create time range
peak_data["time_range"] = (
    peak_data["start_second"].apply(seconds_to_time)
    + " - " +
    peak_data["end_second"].apply(seconds_to_time)
)

# peak score formula
peak_data["peak_score"] = (
    peak_data["comment_count"] * 3
    +
    peak_data["total_likes"] * 2
    +
    peak_data["total_replies"] * 3
    +
    abs(peak_data["avg_sentiment"]) * 10
)

# sort strongest peaks first
peak_data = peak_data.sort_values(
    by="peak_score",
    ascending=False
)

# round values
peak_data["avg_sentiment"] = peak_data["avg_sentiment"].round(2)

peak_data["peak_score"] = peak_data["peak_score"].round().astype(int)

print("TOP VIDEO PEAKS:\n")

display(
    peak_data[
        [
            "time_range",
            "comment_count",
            "total_likes",
            "total_replies",
            "avg_sentiment",
            "peak_score"
        ]
    ].head(10)
)


# =========================================================
# SENTIMENT COUNTS
# =========================================================

print("\n=========== SENTIMENT COUNTS ===========\n")

display(
    df_model["label"].value_counts().to_frame()
)


# =========================================================
# AVERAGE PREDICTION CONFIDENCE
# =========================================================

print(
    "\n=========== AVERAGE PREDICTION CONFIDENCE ===========\n"
)

print(
    round(
        df_model["confidence"].mean(),
        2
    )
)


# =========================================================
# FINAL DATA SIZE
# =========================================================

print("\n=========== FINAL DATA SIZE ===========\n")

print("Raw comments:", len(df_raw))
print("Timestamp comments:", len(df_extracted))
print("Final processed comments:", len(df_model))


# =========================================================
# EXPORT CSV FILES
# =========================================================

df_raw.to_csv(
    "raw_data.csv",
    index=False
)

df_cleaned.to_csv(
    "cleaned_data.csv",
    index=False
)

df_model.to_csv(
    "final_model_output.csv",
    index=False
)

print("\nCSV FILES EXPORTED SUCCESSFULLY")

TOTAL COMMENTS:
42412

=========== RAW COMMENTS SAMPLE ===========

2. Men will do nothing unless they have to, but once they have to, they will do anything"

3. I only watched this interview to try and understand why anyone finds  Tucker appealing and I still don't understand. He'...

4. i gave this video a thymbs up for tucker and the subject matter.  calling zelensky a hero for staying un kyiv is hardcor...

5. Lex shame on you giving the platform this anisemite rat.

6. I mean the war has been a massive failure for Russia in retrospect lol

7. Why did I need to notice this at 1.30 a.m   on a Monday morning? 😫

8. Ugh, I can't listen to this interview. Booo, Boring and lies, just full of crap

9. I'm not this
I'm not 
 that 
It's not that.
It's not
That
Pretty soon
Nothing is
Anything.

10. 1:02:44 Canada does have freedom of speech if aren’t an moron

11. 57:15 no I don’t have $800. Russia is one of favorite places I’d like to visit in my life


COMMENTS WITH TIMESTAMPS:
2008

====

,comment,likes,replies,timestamp,seconds,date
0,Thank you for listening. Here are the timestamps: \r\n0:00 - Introduction\r\n3:53 - Putin\r\n20:07 - Navalny\r\n41:2...,4905,726,0:00,0,02/27/2024
9,1:02:44 Canada does have freedom of speech if aren’t an moron,0,0,1:02:44,3764,05/07/2026
10,57:15 no I don’t have $800. Russia is one of favorite places I’d like to visit in my life,0,0,57:15,3435,05/07/2026
11,55:16 cia owns editing of Wikipedia,0,0,55:16,3316,05/07/2026
14,2:56:49 из за его смеха я хочу знать Английский что бы слушать это в оригинале потому что субтитры пишут одно диктор...,0,0,2:56:49,10609,04/25/2026
18,"2:33:49 Это не та тема, в которую я часто углубляюсь, потому что мне оттуда никто не платит. 😂😂😂",0,0,2:33:49,9229,04/03/2026
33,1:00:00,0,0,1:00:00,3600,02/04/2026
34,44:00,0,0,44:00,2640,02/04/2026
35,10:00,0,0,10:00,600,02/03/2026
51,"2:07:46 до сих пор строят красивые здания, Бурдж-Халифа например. Ну если все здания будут такие то там будут жить т...",0,0,2:07:46,7666,12/30/2025



=========== CLEANED DATA ===========



,comment,clean_comment,likes,replies,timestamp,seconds,date
20365,"53:44 “I bet when you google my picture, 20 years from now, it’ll be a black chick.” I’m dead.","“i bet when you google my picture, 20 years from now, it’ll be a black chick.” i’m dead.",243,9,53:44,3224,03/02/2024
37114,The bit about no one coming to Canada to seek Trudeau's advice had me dying 😂😂😂 2:51:40,the bit about no one coming to canada to seek trudeau's advice had me dying 😂😂😂,136,7,2:51:40,10300,02/27/2024
21491,56:00 ask anyone here in the USA if there is censorship! Come on lex get it together,ask anyone here in the usa if there is censorship! come on lex get it together,95,15,56:00,3360,03/01/2024
19305,1:20:54 Absolutely. Every war eventually ends by sitting down and negotiating. Why in the hell do we have to repeat ...,absolutely. every war eventually ends by sitting down and negotiating. why in the hell do we have to repeat this stu...,55,13,1:20:54,4854,03/02/2024
27092,"“Men do nothing until they have to. But once they have to, they will do anything” - Tucker Carlson 2:55:19","“men do nothing until they have to. but once they have to, they will do anything” - tucker carlson",29,17,2:55:19,10519,02/29/2024
20815,"1:06:37 Norwegian here. We got a journalist reporting actual real news in Ukraine. When studio took over, they corre...","norwegian here. we got a journalist reporting actual real news in ukraine. when studio took over, they corrected his...",48,2,1:06:37,3997,03/02/2024
21534,46:22 😂 Tucker got that Tom Cruise strange/awkward laugh,😂 tucker got that tom cruise strange/awkward laugh,34,10,46:22,2782,03/01/2024
19329,"2:55:28 such a great line: Men do nothing unless they have to, once they have to, they’ll do anything!","such a great line: men do nothing unless they have to, once they have to, they’ll do anything!",39,2,2:55:28,10528,03/02/2024
4503,"1:02:25 Удивительно! Гражданину ""страны со свободой слова"" - Такеру, приходится периодически оправдываться перед ау...","удивительно! гражданину ""страны со свободой слова"" - такеру, приходится периодически оправдываться перед аудиторией ...",35,2,1:02:25,3745,04/09/2024
23664,"1:24:21 ""Bitch are you in charge?"".....lmao. that one got me. Tucker does have some good zingers at times.","""bitch are you in charge?"".....lmao. that one got me. tucker does have some good zingers at times.",34,1,1:24:21,5061,03/01/2024


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...


=========== PEAK DETECTION ===========

TOP VIDEO PEAKS:



,time_range,comment_count,total_likes,total_replies,avg_sentiment,peak_score
103,53:30 - 53:59,5,244,9,0.00,530
302,171:30 - 171:59,3,136,7,-0.67,309
166,89:00 - 89:29,41,43,23,-0.49,283
308,175:00 - 175:29,15,81,19,0.13,265
3,01:30 - 01:59,43,47,12,-0.09,260
108,56:00 - 56:29,2,96,15,-0.50,248
120,62:00 - 62:29,14,60,16,-0.57,216
88,46:00 - 46:29,15,46,15,-0.20,184
150,80:30 - 80:59,5,56,15,-0.80,180
2,01:00 - 01:29,25,27,11,-0.48,167



=========== SENTIMENT COUNTS ===========



,count
label,
negative,798
neutral,567
positive,246



=========== AVERAGE PREDICTION CONFIDENCE ===========

0.75

=========== FINAL DATA SIZE ===========

Raw comments: 42412
Timestamp comments: 2008
Final processed comments: 1611

CSV FILES EXPORTED SUCCESSFULLY
